# Real Judged-Slice Proxy Diagnosis

This is the first time the course leaves the toy world.

Use it like a real debugging check on labeled data that is already cached in `.cache/kayak/`.
The practical question is simple:

- when does a compressed single-proxy path already lose judged quality relative to exact late interaction?

Start with `LIMIT-small` so the miss stays readable, then widen to the cross-slice summary.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "python" / "kayak").exists():
            return candidate
    raise RuntimeError("Run this notebook from the repository or one of its subdirectories.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
sys.path.insert(0, str(REPO_ROOT / "python"))

import kayak
from kayak_bridge.judged_metrics import summarize_ranked_task

CACHE_ROOT = REPO_ROOT / ".cache" / "kayak"
print("Working from repo root:", REPO_ROOT)
print("Using cached tasks from:", CACHE_ROOT)
print("Backends available here:", kayak.available_backends())

In [ ]:
TASK_PATHS = {
    "limit_small_real_subset": CACHE_ROOT / "limit_small_real_subset" / "python_task.json",
    "bright_stackoverflow_real_subset": CACHE_ROOT / "bright_stackoverflow_real_subset" / "python_task.json",
    "legal_rag_bench_real_subset": CACHE_ROOT / "legal_rag_bench_real_subset" / "python_task.json",
    "r2med_biology_real_subset": CACHE_ROOT / "r2med_biology_real_subset" / "python_task.json",
}


def load_task(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(
            f"missing cached task JSON: {path}\n"
            "Rebuild it with: PYTHONPATH=python ./.venv/bin/python python/scripts/build_task_json.py --dataset-key ..."
        )
    return json.loads(path.read_text())


def build_index(task: dict) -> kayak.LateIndex:
    return kayak.documents(
        [row["doc_id"] for row in task["documents"]],
        [np.asarray(row["vectors"], dtype=np.float32) for row in task["documents"]],
        texts=[row["text"] for row in task["documents"]],
    ).pack()


def query_from_row(row: dict) -> kayak.LateQuery:
    return kayak.query(np.asarray(row["vectors"], dtype=np.float32), text=row["text"])


In [ ]:
rows = []
analysis_cache = {}

for slice_name, path in TASK_PATHS.items():
    task = load_task(path)
    index = build_index(task)

    exact_ranked = []
    proxy_full_ranked = []
    proxy_budget8_ranked = []
    per_query = []

    for row in task["queries"]:
        query = query_from_row(row)
        exact = kayak.search_with_plan(
            query,
            index,
            kayak.exact_full_scan_search_plan(
                final_k=task["k"],
                candidate_k=len(task["documents"]),
            ),
            backend=kayak.NUMPY_REFERENCE_BACKEND,
        )
        proxy_full = kayak.search_with_plan(
            query,
            index,
            kayak.document_proxy_search_plan(
                final_k=task["k"],
                candidate_k=task["k"],
            ),
            backend=kayak.NUMPY_REFERENCE_BACKEND,
        )
        proxy_budget8 = kayak.search_with_plan(
            query,
            index,
            kayak.document_proxy_search_plan(
                final_k=task["k"],
                candidate_k=task["k"],
                query_vector_budget=8,
                document_vector_budget=8,
            ),
            backend=kayak.NUMPY_REFERENCE_BACKEND,
        )

        exact_ranked.append(tuple(hit.doc_id for hit in exact.hits))
        proxy_full_ranked.append(tuple(hit.doc_id for hit in proxy_full.hits))
        proxy_budget8_ranked.append(tuple(hit.doc_id for hit in proxy_budget8.hits))
        per_query.append(
            {
                "query_id": row["query_id"],
                "query_text": row["text"],
                "relevant_doc_ids": tuple(row["relevant_doc_ids"]),
                "exact_hits": tuple(hit.doc_id for hit in exact.hits),
                "proxy_full_hits": tuple(hit.doc_id for hit in proxy_full.hits),
                "proxy_budget8_hits": tuple(hit.doc_id for hit in proxy_budget8.hits),
            }
        )

    exact_summary = summarize_ranked_task(task=task, ranked_doc_ids_by_query=exact_ranked)
    proxy_full_summary = summarize_ranked_task(task=task, ranked_doc_ids_by_query=proxy_full_ranked)
    proxy_budget8_summary = summarize_ranked_task(task=task, ranked_doc_ids_by_query=proxy_budget8_ranked)

    rows.append(
        {
            "slice_name": slice_name,
            "primary_metric": task["primary_metric"],
            "exact": exact_summary.primary_value,
            "proxy_full": proxy_full_summary.primary_value,
            "proxy_budget8": proxy_budget8_summary.primary_value,
            "exact_recall_at_k": exact_summary.mean_recall_at_k,
            "proxy_full_recall_at_k": proxy_full_summary.mean_recall_at_k,
            "proxy_budget8_recall_at_k": proxy_budget8_summary.mean_recall_at_k,
        }
    )
    analysis_cache[slice_name] = {
        "task": task,
        "per_query": per_query,
        "index": index,
    }

summary_rows = rows
summary_rows

Interpretation:

- exact late interaction is stronger on all four cached slices in this local comparison
- `LIMIT-small` is the easiest place to feel the miss with your own eyes because the gap is large and the corpus is still inspectable
- the `budget=8` proxy is not monotonically worse than the full-budget proxy on every slice, so proxy compression has to be measured rather than assumed


In [ ]:
limit_task = analysis_cache["limit_small_real_subset"]["task"]
limit_index = analysis_cache["limit_small_real_subset"]["index"]
doc_by_id = {row["doc_id"]: row for row in limit_task["documents"]}

interesting = None
for row in analysis_cache["limit_small_real_subset"]["per_query"]:
    exact_set = set(row["exact_hits"][: limit_task["k"]])
    proxy_set = set(row["proxy_full_hits"][: limit_task["k"]])
    if exact_set != proxy_set:
        interesting = row
        break

interesting

In [ ]:
if interesting is None:
    raise RuntimeError("No differing LIMIT query found.")

print("Inspecting query id:", interesting["query_id"])
print("Question:", interesting["query_text"])
print("Judged relevant docs:", interesting["relevant_doc_ids"])
print("Top-5 if I keep exact late interaction:", interesting["exact_hits"][:5])
print("Top-5 from the compressed proxy path:", interesting["proxy_full_hits"][:5])


In [ ]:
def show_doc(doc_id: str, *, limit: int = 900) -> None:
    print(f"Document: {doc_id}")
    print(doc_by_id[doc_id]["text"][:limit])
    print()


for doc_id in interesting["relevant_doc_ids"]:
    show_doc(doc_id)

for doc_id in interesting["proxy_full_hits"][:5]:
    if doc_id not in interesting["relevant_doc_ids"]:
        show_doc(doc_id)


## Practical Takeaway

This is the real-data version of the same debugging loop:

- start with exact late interaction on the cached task vectors
- compare it against the cheaper proxy path
- inspect where judged quality disappears
- do not assume the proxy heuristic is stable just because it looked fine once

That is the course-grade message:

- use Kayak to verify the failure boundary before you optimize around it
